# Workshop: sentiment analysis 

## Program

- What is natural language processing and what are classification tasks 😀
- NLP tasks

### Build a sentiment classifier
- Load and explore your data
- Text preprocessing
- Load a model in the code environment
- Step-by-step building a classifier with a pre-trained model
- Run classification task: sentiment analysis on a data sample


Other classification datasets for classification tasks:

- [Spam detection dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

- [Hate Speech ETHOS dataset](https://paperswithcode.com/dataset/ethos)

- [Fake News Seek Truth dataset](https://www.kaggle.com/datasets/paulinepeps/truth-seeker-dataset-2023-truthseeker2023)

Some NLP tasks applied to cybersecurity:

- Named Entity Recognition (NER): Identifying entities such as names, locations, organizations, and dates. [CyNER](https://github.com/aiforsec/CyNER)

- Topic Modeling: Discovering hidden topics and themes in large datasets. [BERT Topic](https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html)

Libraries we use:

- [Transformers](https://huggingface.co/docs/transformers/index)

- [Datasets](https://huggingface.co/docs/datasets/index)

- [Torch](https://pytorch.org/)

In [1]:
!pip install transformers datasets -q

In [2]:
'''
We import transformers pipeline and torch
'''

from transformers import pipeline
import torch
from pprint import pprint


## Natural language processing tasks

### An example of previous generation of language model GPT-2

In [3]:
'''
Here we create our first pipeline with the library transformers
'''

from transformers import set_seed
generator = pipeline('text-generation', model='gpt2', clean_up_tokenization_spaces=True)

OSError: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like gpt2 is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
generator("I am a unicorn in a financial office,", max_length=20, truncation=True, num_return_sequences=5)

In [ ]:
generator("To bake cookies I need,", max_length=25, num_return_sequences=2, truncation=True)

In [ ]:
generator("I don't like cats,", max_length=20, num_return_sequences=5, truncation=True)

In [ ]:
#generator("...", max_length=.., num_return_sequences=..)

In [ ]:
#generator()

### Labeling

In [ ]:
from transformers import pipeline

classifier_labels = pipeline(model="facebook/bart-large-mnli")

In [ ]:
results = classifier_labels(
    "I have a problem with my iphone that needs to be resolved !!",
    candidate_labels=["urgent", "not urgent", "phone", "tablet", "computer"],
)

for i, score in enumerate(results['scores']):
  results['scores'][i] = round(score, 2)

In [ ]:
results

In [ ]:
#change the sentence and print the results:

## Build a sentiment analysis classifier

### Instantiate a pipeline

In [ ]:
from transformers import pipeline

classifier_sentiments = pipeline("sentiment-analysis")

### Run the classifier

In [ ]:
results = classifier_sentiments(["This is cool","This is not that cool", "This is bad"])
for result in results:
    result['raw'] = result['score']
    result['score'] = round(result['score'], 2)
results

### Many models for sentiment classification

By default transformers library uses a DistilBERT model for the pipelines we have created. Other models are available for the same type of task:

- [Roberta sentiment](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)

- [BERT base personality](https://huggingface.co/Minej/bert-base-personality)

Next, let's import this model into our code:

## Tokenizer

### What is a tokenizer

- Tokenization is the process of breaking down text into smaller **units** called **tokens**. In order to process text the computer needs first to transform it into numbers.

- Tokens are the basic building blocks used by Transformers models to understand and process text.

- Tokens can represent **words, subwords, or even individual characters**, depending on the model's vocabulary.

### Instanciate a tokenizer

The model we use is a [BERT](https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment)

In [ ]:
from transformers import DistilBertTokenizer

model = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = DistilBertTokenizer.from_pretrained(model)


We add our tokenizer to our pipeline:


In [ ]:
new_classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

In [ ]:
new_classifier("I am happy")


## Tokenization

A token is a value extracted from a **vocabulary list**.

A vocabulary list is a set words.

## Create tokens

### Split method

In [ ]:
tokenized_text = "We are at a hacking conference.".split()
print(tokenized_text)

In [ ]:
#tokenize another sentence with split method.

### Use a tokenizer

In [ ]:
sequence = "We are at a hacking conference."
tokens = tokenizer.tokenize(sequence)

print(tokens)

In [ ]:
#What is different?

In [ ]:
#create a new sequence and generate tokens for this sequence

### Try another tokenizer

In [ ]:
from transformers import XLNetTokenizer


another_tokenizer = XLNetTokenizer.from_pretrained("xlnet/xlnet-base-cased")
new_tokens = another_tokenizer.tokenize(sequence)


In [ ]:
print(f"Tokens: {new_tokens}\n")

In [ ]:
#What is different?

## Input IDs

Remember our sentence : "We are at a hacking conference." Let's see token ids for this sentence.

In [ ]:
'''
Our current tokens:
Tokens: ['▁We', '▁are', '▁at', '▁a', '▁hacking', '▁conference', '.']
'''

ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

In [ ]:
# Tokenize another sentence and see the token ids:

# your_sentence = ""
# your_tokens =
# your_ids =

In [ ]:
# @title
your_sentence = "We are at a hackers party."
your_tokens = tokenizer.tokenize(your_sentence)
your_ids = tokenizer.convert_tokens_to_ids(your_tokens)

In [ ]:
# @title
print(your_tokens)
print(your_ids)

## Padding and truncation

Language models work with **tensors**, we need them to be **the same length**.

```
padding=True and truncation=True
```

In [ ]:
sentences = ["A white poney.", "A white poney in the garden."]

batch = tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt") #pt for pyTorch

In [ ]:
pprint(batch)

In [ ]:
#What are the ```'101'``` and ```'102'``` in the token list?


In [ ]:
#what are the zeros?

In [ ]:
#Try out with two new sentences

Note it returns a dictionary with keys ```'input_ids'``` and ```'attention_mask'```, with two tensors the 'input ids' tensor and the 'attention_mask' tensor.
input_ids are unique ids.

# Dataset

We use the Dataset object from huggingface: https://huggingface.co/docs/datasets/v2.1.0/en/access 

## Load a dataset from the hub

In [ ]:
from datasets import load_dataset

dataset = load_dataset("carblacac/twitter-sentiment-analysis", split="train")

In [ ]:
dataset

The labels here are ```'feeling'``` let's change this for **sentiment**


In [ ]:
# Rename the column
dataset = dataset.rename_column('feeling', 'sentiment')

# Verify the column has been renamed
pprint(dataset.features)

In [ ]:
# Explore the dataset
dataset[0]

In [ ]:
sample = dataset["text"][:10]
sample

In [ ]:
print(dataset.info)

In [ ]:
print(dataset.features)

In [ ]:
#How many tweets are in this dataset?
#What is the title of the dataset?
#When has it been published?

In [ ]:
import pandas as pd

# Convert the dataset to a pandas DataFrame
df = pd.DataFrame(dataset)

# Display the columns of the DataFrame
print(df.columns)

## Tokenize the dataset

In [ ]:
tokenizer(dataset[0]["text"])


In [ ]:
def tokenization(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(tokenization, batched=True)

In [ ]:
tokenized_dataset

Now your set is ready for training!

## Create sample of the dataset

In [ ]:
import pandas as pd

In [ ]:
df = dataset.to_pandas()
sample = df.head(10)

In [ ]:
sample

In [ ]:
df

In [ ]:
#Show the first 4 tweets of this dataframe

In [ ]:
# @title
df[:4]

In [ ]:
#Show only the labels for the first 4 tweets

## Classify sentiment

![Pipeline](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)

Source: HuggingFace NLP course

In [ ]:
classifier = pipeline("sentiment-analysis")

def predict_sentiment(text):
  result = classifier(text)[0]
  return result['label']

sentiment = []
for text in sample['text']:
  sentiment.append(predict_sentiment(text))

In [ ]:

sample['predicted_sentiment'] = sentiment
pprint(sentiment)
sample

In [ ]:
# let's remap the labels to 0 for negative and 1 for positive:

# Define the mapping function
def map_sentiment(preds):
    return 1 if preds == "positive" else 0

for i in range(len(sample)):
    if sample.loc[i, 'predicted_sentiment'] == "POSITIVE":
        sample.loc[i, 'predicted_sentiment'] = 1
    else:
        sample.loc[i, 'predicted_sentiment'] = 0
        
# Use pprint to print the dataset features
sample

In [ ]:
#let's compare the predictions with the actual labels
sample

In [ ]:
# Initialize counters
correct_predictions = 0
incorrect_predictions = 0

# compare predictions with actual labels
for i in range(len(sample)):
    if sample.loc[i, 'sentiment'] == sample.loc[i, 'predicted_sentiment']:
        correct_predictions += 1
    else:
        incorrect_predictions += 1

# Calculate percentages
total_predictions = correct_predictions + incorrect_predictions
correct_percentage = (correct_predictions / total_predictions) * 100
incorrect_percentage = (incorrect_predictions / total_predictions) * 100


# Print the results
print(f"Correct predictions: {correct_predictions}")
print(f"Incorrect predictions: {incorrect_predictions}")

# Thanks!

### Contact: [Twitter](www.twitter.com/hello_locked)  | [Mastdodon](https://infosec.exchange/deck/@C00kie_two)